<a href="https://colab.research.google.com/github/robbarto2/GenAI-Foundations/blob/main/Instruction_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## What is Unsloth?

**Run on Colab with GPU runtime.** Unsloth only supports NVIDIA, AMD, and Intel GPUs. If you see *"Unsloth currently only works on NVIDIA, AMD and Intel GPUs"*, you're running locally without a supported GPU—use **Google Colab** and set the runtime to GPU.

**Unsloth** is an optimized library for fine-tuning large language models. Key benefits:

- **~2x faster training** — Fused kernels and memory optimizations speed up each step.
- **~70% less VRAM** — You can train on free Colab or use larger models and batch sizes.

This notebook uses a **3B Llama** model so you can see the difference: training 3B on a free Colab T4 is practical with Unsloth; with the standard transformers + PEFT stack it is often tight or runs out of memory.

### Why we use Unsloth here

This demo uses Unsloth instead of plain transformers + PEFT to show **faster, more memory-efficient** fine-tuning. You get the same kind of instruction fine-tuning with less wait and lower GPU memory use—ideal for teaching and for running on free Colab.

In [2]:
# Unsloth: Colab-friendly install (run with GPU runtime, restart if recommended)
# Use stable pip install to avoid KeyError: 'sanitize_logprob' (git+trl version mismatch)
!pip install unsloth -q
!pip install "trl<0.15.0" --force-reinstall --no-deps -q 2>/dev/null || true
!pip install --no-deps peft accelerate bitsandbytes xformers datasets -q



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Import the necessary libraries

In [ ]:
import torch
import os
import time
try:
    from unsloth import FastLanguageModel
    from datasets import Dataset, DatasetDict
    from trl import SFTConfig, SFTTrainer
except NotImplementedError as e:
    if "only works on NVIDIA" in str(e):
        raise RuntimeError(
            "Unsloth requires an NVIDIA, AMD, or Intel GPU. "
            "Run this notebook on Google Colab with GPU (Runtime → Change runtime type → T4 GPU)."
        ) from e
    raise

os.environ["WANDB_DISABLED"] = "true"

Log into Hugging Face with a secure token

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Hugging Face token not found in Colab secrets.")

login(token=hf_token)
import os
os.environ["HF_TOKEN"] = hf_token  # for model load later
print("Successfully logged into Hugging Face!")

Create Some Instruction Training Examples to Fine-Tune the Model

In [ ]:
examples = []

In [ ]:
examples = [
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I haven't received my order #12345 yet. It's been a week.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Delayed
Reference: #12345
Action Taken: Order status check initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I received the wrong item in my order #67890.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Wrong Item Delivered
Reference: #67890
Action Taken: Replacement item initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #98765 is damaged. What should I do?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Damaged Item
Reference: #98765
Action Taken: Return and replacement process initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I want to cancel my order #54321.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Cancellation Request
Reference: #54321
Action Taken: Cancellation process initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I was charged twice for my order #11223. Please help.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Billing Issue
Reference: #11223
Action Taken: Refund request initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "When will my order #44556 arrive?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: In Transit
Reference: #44556
Action Taken: Estimated delivery date provided

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #99988 was returned. When will I get my refund?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Returned
Reference: #99988
Action Taken: Refund process initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I can't track my order #66777. Can you help?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Tracking Issue
Reference: #66777
Action Taken: Tracking link resent

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
        {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #56789 was supposed to arrive yesterday, but I haven't received it.",
        "output": """Thank you for contacting Tech Support.

Order Status: Delayed
Reference: #56789
Action Taken: Initiated a follow-up with the shipping carrier

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I received a damaged item in my order #98765. What should I do?",
        "output": """Thank you for contacting Tech Support.

Order Status: Damaged Item
Reference: #98765
Action Taken: Replacement process initiated. Return instructions sent.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I was charged twice for my order #11223. Please help.",
        "output": """Thank you for contacting Tech Support.

Order Status: Billing Issue
Reference: #11223
Action Taken: Refund process initiated for duplicate charge

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "When will my order #33445 arrive?",
        "output": """Thank you for contacting Tech Support.

Order Status: In Transit
Reference: #33445
Action Taken: Estimated delivery date provided via email

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I need to cancel my order #44556 immediately.",
        "output": """Thank you for contacting Tech Support.

Order Status: Cancellation Requested
Reference: #44556
Action Taken: Cancellation process initiated. Confirmation email sent.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I can't track my order #77889. The tracking link doesn't work.",
        "output": """Thank you for contacting Tech Support.

Order Status: Tracking Issue
Reference: #77889
Action Taken: Sent updated tracking link. Contacted carrier for more information.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I returned my order #99000 last week. When will I get my refund?",
        "output": """Thank you for contacting Tech Support.

Order Status: Returned
Reference: #99000
Action Taken: Refund process initiated. Refund expected within 5 business days.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #11122 arrived, but it's missing some items.",
        "output": """Thank you for contacting Tech Support.

Order Status: Incomplete Order
Reference: #11122
Action Taken: Missing items identified and replacement initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    }
]


Format our training examples the way we want (Instruction, Input, Response)

In [ ]:
dataset = Dataset.from_list(examples)

In [ ]:
def format_example(example):
    instruction = example["instruction"]
    input_text = example["input"]
    output = example["output"]
    return {
        "text": f"### Instruction: {instruction}\n### Input: {input_text}\n### Response: {output}"
    }

formatted_examples = [format_example(ex) for ex in examples]
dataset = Dataset.from_list(formatted_examples)

Load the 3B model with Unsloth

**What happens when we load with Unsloth:** `FastLanguageModel` loads the model with Unsloth's optimized kernels and memory layout. We still add LoRA (or QLoRA with `load_in_4bit=True`) on top for parameter-efficient training.

In [ ]:
# What happens here: FastLanguageModel loads the model with Unsloth's optimized kernels
# and memory layout. We add LoRA (or QLoRA with load_in_4bit=True) for parameter-efficient training.

model_name = "unsloth/Llama-3.2-3B-Instruct"  # 3B model for demo; use load_in_4bit=True on T4 for headroom
max_seq_length = 512
hf_token = os.getenv("HF_TOKEN")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,  # None = auto-detect best dtype for your GPU
    load_in_4bit=True,  # QLoRA: fits 3B on Colab T4 comfortably
    token=hf_token,
)

print("Model and tokenizer loaded with Unsloth!")

In [ ]:
# Ensure tokenizer has a padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use eos_token if available
    # Or add a new pad token:
    # tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [ ]:
# Split into train/validation (80/20). SFTTrainer will tokenize using dataset_text_field="text".
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Train: {len(train_dataset)} examples, Eval: {len(eval_dataset)} examples")

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

['text', 'input_ids', 'attention_mask', 'labels']
Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 16
})


In [ ]:
# Add LoRA adapters (Unsloth's fast path)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# SFTTrainer config (Unsloth works with TRL's SFTTrainer)
training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=30,
    per_device_train_batch_size=2,  # 3B on T4: use 2 with QLoRA
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    max_seq_length=max_seq_length,
    report_to="none",
)

In [ ]:
# Optional: check VRAM before training (run !nvidia-smi in another cell to compare after)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits 2>/dev/null || echo "nvidia-smi not available"

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 12
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
})


In [ ]:
# Unsloth: print trainable parameters
model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


**What Unsloth does during training:** Unsloth's training path uses fused kernels and less memory, so each step is faster and peak VRAM is lower than with the standard Hugging Face Trainer + PEFT. That's why we can run a 3B model on a free Colab T4.

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
)

In [ ]:
train_start = time.time()
trainer.train()
train_elapsed = (time.time() - train_start) / 60

trainer.save_model("./final_model")
print("Training complete and model saved!")
print(f"With Unsloth this run took {train_elapsed:.1f} min. Typical speedup: ~2x, ~70% less VRAM vs standard stack.")

Step,Training Loss,Validation Loss
20,No log,3.390074
40,4.433400,0.468085
60,0.392600,0.187010
80,0.392600,0.124155


Training complete and model saved!


**Recap:** With Unsloth we trained a 3B model on Colab in the time above. The same run with the standard transformers + PEFT stack would typically use more VRAM and take longer—Unsloth’s optimizations make the difference visible at this scale.

Inference function

In [ ]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """
    Generate a response using the fine-tuned model.

    Args:
        model: The fine-tuned model
        tokenizer: The tokenizer
        instruction: The instruction for the model
        input_text: Optional input text

    Returns:
        str: The generated response
    """
    # Format input to match training format
    prompt = f"### Instruction: {instruction}\n### Input: {input_text}\n### Response:"

    # Tokenize the prompt
    inputs = tokenizer(prompt,
                      return_tensors="pt",
                      truncation=True,
                      max_length=512,
                      add_special_tokens=True)

    # Move inputs to the same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,  # Adjust based on desired response length
        num_return_sequences=1,
        temperature=0.1,     # Adjust for response creativity (0.0-1.0)
        do_sample=True,
        top_p=0.95,         # Nucleus sampling
        top_k=50,           # Top-k sampling
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Decode the response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the response part (after "### Response:")
    response_parts = response.split("### Response:")
    if len(response_parts) > 1:
        response = response_parts[1].strip()

    return response


In [ ]:
def post_process_response(response):
    """
    Post-process the model's response to ensure it adheres to the desired format.

    Args:
        response (str): The raw response from the model.

    Returns:
        str: The post-processed response with required sections.
    """
    # Define required sections
    sections = ["Order Status", "Reference", "Action Taken", "Need immediate assistance?"]
    processed_response = response.strip()

    # Ensure all required sections exist
    for section in sections:
        if section not in processed_response:
            processed_response += f"\n{section}: [Details Missing]"

    return processed_response



In [ ]:
# Example input
instruction = "Respond to this customer inquiry following our format"
input_text = "I haven't received my order #12345 yet. It's been a week."

raw_response = generate_response(model, tokenizer, instruction, input_text)
final_response = post_process_response(raw_response)

# Print the results
print("\nINSTRUCTION:")
print(instruction)
print("\nINPUT:")
print(input_text)
print("\nFINAL RESPONSE:")
print(final_response)


INSTRUCTION:
Respond to this customer inquiry following our format

INPUT:
I haven't received my order #12345 yet. It's been a week.

FINAL RESPONSE:
Thank you for contacting TechCorp Support.

Order Status: Delayed
Reference: #12345
Action Taken: Tracking update sent
Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support


In [ ]:
# Check training logs/loss
print("Training logs from the last few steps:")
print(trainer.state.log_history[-5:])  # Shows the last 5 training logs

Training logs from the last few steps:
[{'train_runtime': 9.3961, 'train_samples_per_second': 4.257, 'train_steps_per_second': 1.064, 'total_flos': 119789573898240.0, 'train_loss': 2.6833833694458007, 'epoch': 10.0, 'step': 10}]
